In [ ]:
%reload_ext watermark
%matplotlib inline

import os
import pandas
from metapool.sample_sheet import (
    make_sample_sheet, PACBIO_ABSQUANT_SHEET_TYPE, PACBIO_METAG_SHEET_TYPE,
    _ASSAY_KEY, _SHEET_TYPE_KEY, _SHEET_VERSION_KEY, _METAGENOMIC,
    _BIOINFORMATICS_KEY, _CONTACT_KEY, _EXPERIMENT_NAME_KEY,
    _SAMPLE_CONTEXT_KEY, SS_SAMPLE_ID_KEY)
from metapool.controls import \
    get_delimited_controls_details_from_compressed_plate, PM_SAMPLE_KEY, \
    PM_PROJECT_NAME_KEY, PM_PROJECT_PLATE_KEY
from metapool.mp_strings import (
    PM_WELL_KEY, BARCODE_ID_KEY, PM_BLANK_KEY, SYNDNA_POOL_MASS_NG_KEY,
    ELUTION_VOL_KEY, EXTRACTED_GDNA_CONC_KEY, SYNDNA_POOL_NUM_KEY,
    TWIST_ADAPTOR_ID_KEY, SYNDNA_IS_TWISTED_KEY)
from metapool.metapool import bcl_scrub_name

%watermark -i -v -iv -m -h -p metapool,sample_sheet,openpyxl -u

In [ ]:
! conda list

<small>Amanda Birmingham, 2025-11</small>

# Temporary PacBio Sample Sheet Builder

This notebook produces a pacbio sample sheet from a .csv-format file download of a google sheet processing doc.

### Step 1: Provide inputs

In [ ]:
## INPUT
# processing_doc_fp = './Prep_information_ZenglerSoil_Aug_2025 - Sheet1_rep2.csv'
# processing_doc_fp = '/Users/amandabirmingham/Desktop/Prep_information_TestSoil_Aug_2025 - Sheet1.csv'
processing_doc_fp = '/Users/amandabirmingham/Desktop/Prep_information_AmpliFiValidation_Aug_2025 - Prep_information.csv'

# IMPORTANT: All these keys must match exactly the column headers in the
# processing doc shown above.  They seem to change from doc to doc,
# these must be updated EVERY TIME.
# qiita_id_key = "Qiita_ID"
# sample_name_col_key = "Sample_ID"
# sample_type_key = "sample_type"
# plate_col_key = "Extraction_Plate"
# project_name_col_key = "Project_Name"
# well_col_key = "Library_Well_ID"
# twist_adaptor_id_col_key = "Twist_UDI_Barcode"
# barcode_col_key = "Barcode"
# run_id_key = "Sequencing Run"
# run_title_key = "Sequencing Run Name"

qiita_id_key = "Qiita_ID"
sample_name_col_key = "sample_name"
sample_type_key = "sample_type"
plate_col_key = "Extraction_Plate"
project_name_col_key = "Project_Name"
well_col_key = "Library_Well_ID"
twist_adaptor_id_col_key = "Amplification_Twist_UDI"
barcode_col_key = "Barcode"
run_id_key = "Sequencing Run"
run_title_key = "Sequencing Run Name"

# output_dir = "./sample_sheets"
output_dir = '/Users/amandabirmingham/Desktop'
output_fname = None # if None, will be generated from run id


In [ ]:
# DEFAULTS: DO NOT CHANGE THESE
# unless you definitely know what you are doing
lib_const_protocol = "Knight Lab PacBio"
experimental_design_desc = "fecal samples for metagenomic sequencing"
sheet_type = PACBIO_METAG_SHEET_TYPE

In [ ]:
# CONSTANTS: DO NOT CHANGE THESE EITHER
# unless you definitely know what you are doing
ISOLATE_STR = "isolate"
SHEET_VERSION_NUM = "11"
ASSUMED_SYNDNA_POOL_NUM = "2000"
EXPECTED_ABSQUANT_COLS = [
    SYNDNA_POOL_MASS_NG_KEY,    # mass_syndna_input_ng
    ELUTION_VOL_KEY,            # vol_extracted_elution_ul
    EXTRACTED_GDNA_CONC_KEY,    # extracted_gdna_concentration_ng_ul
    SYNDNA_IS_TWISTED_KEY       # syndna_is_twisted
]
# NB: calc_mass_sample_aliquot_input_g is needed to do the actual abs quant,
# although not for the sample sheet generation

### Step 2: Load and validate processing info

In [ ]:
# load the processing google sheet csv as a dataframe
processing_df = pandas.read_csv(processing_doc_fp, dtype=str)
processing_df.shape

In [ ]:
# make a working copy of the info
info_df = processing_df.copy()

# drop any column that contains only "N/A" values
info_df = info_df.loc[:, (info_df != "N/A").any(axis=0)]
# drop any column that contains only NaN values
info_df = info_df.dropna(axis=1, how='all')

In [ ]:
# determine whether this is an abs quant prep
is_absquant = False
# if there is a column named "Absolute Quantification", assume it is definitive
if "Absolute Quantification" in info_df.columns:
    abs_quant_vals = set(info_df["Absolute Quantification"].values)
    if "TRUE" in abs_quant_vals:
        is_absquant = True
else:
    # if any expected abs quant columns are present, assume abs quant
    for col in EXPECTED_ABSQUANT_COLS:
        if col in info_df.columns:
            is_absquant = True
            break

is_absquant

In [ ]:
# determine which columns are required and
cols_to_get = [sample_name_col_key, plate_col_key, well_col_key,
               barcode_col_key, twist_adaptor_id_col_key, project_name_col_key,
               qiita_id_key, sample_type_key]
if PM_BLANK_KEY in info_df.columns:
    cols_to_get.append(PM_BLANK_KEY)

# check if any required columns are missing
if is_absquant:
    cols_to_get.extend(EXPECTED_ABSQUANT_COLS)
missing_cols = [
    col for col in cols_to_get if col not in info_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

In [ ]:
# drop any row that doesn't have a run id
info_df = info_df.dropna(subset=[run_id_key])

# eventually may make a sample sheet for each different run id in the
# processing doc, but for now just error if more than one
curr_run_ids = set(list(info_df[run_id_key].values))
if len(curr_run_ids) >1:
    raise ValueError(f"Multiple run ids found: {curr_run_ids}")

curr_run_id = list(curr_run_ids)[0]
curr_run_title = info_df[run_title_key].values[0]
curr_run_id

In [ ]:
# drop any row that doesn't have a qiita id
info_df = info_df.dropna(subset=[qiita_id_key])

# get all the unique qiita ids in the curr_run_df
qiita_ids = set(list(info_df[qiita_id_key].values))
qiita_ids

In [ ]:
# drop any rows with missing values in the sample_name_col_key
info_df = info_df.dropna(subset=[sample_name_col_key])
info_df.shape

In [ ]:
# ensure that each unique value for the project_name_col_key
# is associated with one and only one qiita id and vice versa
project_to_qiita = {}
qiita_to_project = {}
for _, row in info_df.iterrows():
    proj = row[project_name_col_key]
    qiita_id = row[qiita_id_key]
    if proj in project_to_qiita:
        if project_to_qiita[proj] != qiita_id:
            raise ValueError(
                f"Project name '{proj}' is associated with multiple Qiita IDs")
    else:
        project_to_qiita[proj] = qiita_id
    if qiita_id in qiita_to_project:
        if qiita_to_project[qiita_id] != proj:
            raise ValueError(
                f"Qiita ID '{qiita_id}' is associated with "
                f"multiple project names")
    else:
        qiita_to_project[qiita_id] = proj
# next row in info_df

### Step 3: Standardize processing info

In [ ]:
# strip leading/trailing whitespace from all entries
info_df = info_df.map(
    lambda x: x.strip() if isinstance(x, str) else x)

# replace any spaces in contents of the pacbio barcode column with nothing
info_df[barcode_col_key] = \
    info_df[barcode_col_key].str.replace(" ", "")

# replace any spaces in the contents of the plate or
# project columns with underscores
info_df[plate_col_key] = \
    info_df[plate_col_key].str.replace(" ", "_")
info_df[project_name_col_key] = \
    info_df[project_name_col_key].str.replace(" ", "_")

# replace any nans in the twist_adaptor_id_col_key with empty string
info_df[twist_adaptor_id_col_key] = \
    info_df[twist_adaptor_id_col_key].fillna("")

In [ ]:
# for each row, if the project_name_col_key
# value does not end with the value of the QIITA_ID_KEY, append
# _<qiita id> to it (this ensures that the project name is fully qualified
#  with qiita id)
info_df[project_name_col_key] = info_df.apply(
    lambda row: row[project_name_col_key]
    if row[project_name_col_key].endswith(f"_{row[qiita_id_key]}")
    else f"{row[project_name_col_key]}_{row[qiita_id_key]}",
    axis=1
)

### Step 4: Mock plate_df and metadata dictionary

In [ ]:
# subset the processing info to just the needed columns for the plate_df
plate_df = info_df[cols_to_get].copy()
# note that not all columns in cols_to_get are renamed;
# also note that qiita id and sample type columns are retained but
# they aren't a canonical part of the plate_df
plate_df = plate_df.rename(columns={
    sample_name_col_key: PM_SAMPLE_KEY,
    plate_col_key: PM_PROJECT_PLATE_KEY,
    well_col_key: PM_WELL_KEY,
    barcode_col_key: BARCODE_ID_KEY,
    twist_adaptor_id_col_key: TWIST_ADAPTOR_ID_KEY,
    project_name_col_key: PM_PROJECT_NAME_KEY
})

In [ ]:
# make the sample ids from the sample names by applying bcl_scrub_name
plate_df[SS_SAMPLE_ID_KEY] = \
    plate_df[PM_SAMPLE_KEY].apply(bcl_scrub_name)

In [ ]:
if is_absquant:
    lib_const_protocol += " absquant"
    experimental_design_desc += " with absquant"
    sheet_type = PACBIO_ABSQUANT_SHEET_TYPE
    plate_df[SYNDNA_POOL_NUM_KEY] = ASSUMED_SYNDNA_POOL_NUM

    # set the SYNDNA_IS_TWISTED_KEY column to empty string for anything
    # that doesn't have a twist adaptor id
    def set_syndna_is_twisted(row):
        if row[TWIST_ADAPTOR_ID_KEY] == "":
            return ""
        else:
            return row[SYNDNA_IS_TWISTED_KEY]
    plate_df[SYNDNA_IS_TWISTED_KEY] = \
        plate_df.apply(set_syndna_is_twisted, axis=1)

plate_df.head()

In [ ]:
# auto-generate the sample context info for the run
context_dict_list = \
    get_delimited_controls_details_from_compressed_plate(plate_df)
context_dict_list

In [ ]:
# fill the non-project-specific parts of the metadata dictionary
metadata = {
    _ASSAY_KEY: _METAGENOMIC,
    _SHEET_VERSION_KEY: SHEET_VERSION_NUM,
    _SHEET_TYPE_KEY: sheet_type,
    _EXPERIMENT_NAME_KEY: curr_run_title,
    _BIOINFORMATICS_KEY:  [],
    _CONTACT_KEY: [],
    _SAMPLE_CONTEXT_KEY: context_dict_list
}

### Step 5: Generate project-specific metadata

In [ ]:
# loop over each qiita id and add to the metadata dictionary for it
for curr_qiita_id in qiita_ids:
    # get the subset of plate_df for this qiita id
    curr_qiita_id_mask = plate_df[qiita_id_key] == curr_qiita_id
    curr_qiita_id_df = plate_df.loc[curr_qiita_id_mask].copy()

    # if there is any value other than ISOLATE_STR in curr_sample_types,
    # then we proceed with making the sample sheet
    curr_sample_types = curr_qiita_id_df[sample_type_key].values
    all_isolates = all([x == ISOLATE_STR for x in curr_sample_types])
    if all_isolates:
        print(f"All samples are isolates for Qiita study "
              f"{curr_qiita_id} in run ID {curr_run_id}; "
              f"skipping sample sheet generation.")
        continue
    has_human = any([x.startswith("human") for x in curr_sample_types])

    # get the unique project name(s) for this qiita id
    unique_projects = curr_qiita_id_df[PM_PROJECT_NAME_KEY].unique()
    if len(unique_projects) > 1:
        print(f"Multiple Sample_Project values found for Qiita study "
              f"{curr_qiita_id} in run ID {curr_run_id}: "
              f"{unique_projects}")

    metadata[_BIOINFORMATICS_KEY].append(
        {
            'Sample_Project': unique_projects[0],
            'QiitaID': curr_qiita_id,
            'HumanFiltering': has_human,
            'library_construction_protocol': lib_const_protocol,
            'experiment_design_description': experimental_design_desc,
            'contains_replicates': 'False'
        })

    metadata[_CONTACT_KEY].append(
        {
            'Sample_Project': unique_projects[0],
            'Email': "gail.ackermann50@gmail.com"
        }
    )
# next qiita id

metadata

### Step 6: Generate sample sheet

In [ ]:
pacbio_sheet = make_sample_sheet(metadata, plate_df, "Revio", [1])

In [ ]:
if output_fname is None:
    output_fname = f"{curr_run_id}_sample_sheet.csv"
out_fp = f"{output_dir}/{output_fname}"

In [ ]:
if os.path.isfile(out_fp):
    print("Warning! This file exists already!")

In [ ]:
with open(out_fp, 'w') as out_fh:
    pacbio_sheet.write(out_fh)